# House Price Prediction Model
This notebook covers data loading, cleaning, EDA, modeling, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

os.makedirs("eda_plots", exist_ok=True)
os.makedirs("models", exist_ok=True)

### 1. Load Data

In [ ]:
print("Loading data...")
df = pd.read_csv("data/house_prices.csv")
df.head()

### 2. Cleaning & Feature Engineering

In [ ]:
print("Cleaning data...")
def parse_amount(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    try:
        if "lac" in x:
            return float(x.replace("lac", "").strip()) * 1e5
        if "cr" in x:
            return float(x.replace("cr", "").strip()) * 1e7
        return float(x.replace(",", ""))
    except ValueError:
        return None

df["price_clean"] = df["Amount(in rupees)"].apply(parse_amount)
df = df.dropna(subset=["price_clean"])

def parse_area(x):
    if not isinstance(x, str):
        return None
    x = x.lower().strip()
    try:
        if "sqft" in x:
            return float(x.replace("sqft", "").strip())
        if "sqm" in x:
            return float(x.replace("sqm", "").strip()) * 10.764
        return float(''.join(c for c in x if c.isdigit() or c == '.'))
    except Exception:
        return None

df["carpet_area_sqft"] = df["Carpet Area"].apply(parse_area)

def parse_floor(x):
    if not isinstance(x, str):
        return None
    x = x.lower()
    if "ground" in x:
        return 0
    if "basement" in x:
        return -1
    import re
    match = re.search(r'(\d+)', x)
    if match:
        return int(match.group(1))
    return None

df["floor_num"] = df["Floor"].apply(parse_floor)

for col in ["Bathroom", "Balcony", "Car Parking"]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

top_locations = df["location"].value_counts().nlargest(50).index
df["location_grouped"] = df["location"].apply(lambda x: x if x in top_locations else "Other")

df = df.dropna(subset=["carpet_area_sqft"])
df["price_per_sqft"] = df["price_clean"] / df["carpet_area_sqft"]
q1 = df["price_per_sqft"].quantile(0.01)
q99 = df["price_per_sqft"].quantile(0.99)
df = df[(df["price_per_sqft"] >= q1) & (df["price_per_sqft"] <= q99)]


### 3. EDA

In [ ]:
print("Generating EDA plots...")
plt.figure(figsize=(8, 6))
sns.histplot(df["price_clean"], log_scale=True, kde=True)
plt.title("Price Distribution (Log Scale)")
plt.savefig("eda_plots/price_distribution.png", bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 6))
sns.scatterplot(x=df["carpet_area_sqft"], y=df["price_clean"])
plt.xscale("log")
plt.yscale("log")
plt.title("Price vs. Carpet Area (Log-Log Scale)")
plt.savefig("eda_plots/price_vs_area.png", bbox_inches="tight")
plt.show()


### 4. Modeling Pipeline

In [ ]:
print("Building and training pipeline...")
numeric_features = ["carpet_area_sqft", "floor_num", "Bathroom", "Balcony"]
categorical_features = ["location_grouped", "Furnishing", "Transaction", "Ownership", "facing"]

df[categorical_features] = df[categorical_features].fillna("Unknown")
X = df[numeric_features + categorical_features]
y = df["price_clean"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

lr_model = Pipeline([
    ("prep", preprocessor),
    ("reg", LinearRegression())
])
lr_model.fit(X_train, np.log1p(y_train))
lr_pred = np.expm1(lr_model.predict(X_test))

rf_model = Pipeline([
    ("prep", preprocessor),
    ("reg", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])
rf_model.fit(X_train, np.log1p(y_train))
rf_pred = np.expm1(rf_model.predict(X_test))

def print_metrics(name, y_true, y_pred):
    print(f"--- {name} ---")
    print(f"MAE  : {mean_absolute_error(y_true, y_pred):,.2f}")
    print(f"RMSE : {np.sqrt(mean_squared_error(y_true, y_pred)):,.2f}")
    print(f"R2   : {r2_score(y_true, y_pred):.4f}")

print_metrics("Linear Regression", y_test, lr_pred)
print_metrics("Random Forest", y_test, rf_pred)


### 5. Export

In [ ]:
print("Exporting model and metadata...")
joblib.dump(rf_model, "models/house_price.pkl")

locations = sorted(df["location_grouped"].unique().tolist())
with open("models/locations.json", "w") as f:
    json.dump(locations, f)

print("Done! Model saved to models/house_price.pkl")
